# Week 5 — Optuna + GPU: Bayesian Tuning at Industrial Scale

> *Why grid search is bankrupt, what TPE actually does, and how to drive XGBoost / LightGBM / CatBoost concurrently on GPU with median pruning.*

## Learning objectives

By the end of this notebook, you will be able to:

1. Quantify the inefficiency of grid and random search in high-dimensional spaces.
2. Derive the TPE acquisition function $\ell(\theta) / g(\theta)$ from Expected Improvement.
3. Configure Optuna with TPE sampling and median/Hyperband pruning.
4. Train the three production libraries on GPU with appropriate device flags.
5. Run a multi-algorithm study with persistent SQLite storage.

## Outline

1. **Why grid search and random search fail at high dimensions**
2. **Bayesian optimization** as sequential model-based optimization
3. **The TPE algorithm** — densities $\ell$ and $g$, EI acquisition
4. **Pruners** — median and Hyperband
5. **GPU acceleration paths** for XGBoost, LightGBM, CatBoost
6. **End-to-end study** — concurrent tuning of all three libraries
7. **Persisting and resuming studies** with SQLite storage


## 1. The inefficiency of grid and random search

### Grid search

Suppose you have $d$ hyperparameters, each discretized into $k$ values. Total trials: $k^d$. For 5 parameters with 10 values each, that is $100{,}000$ trials. If each trial is a 2-minute model fit, that is **138 days**.

### Random search (Bergstra & Bengio, 2012)

Random search is provably better than grid search when only some dimensions matter (the "important-dimensions" theorem): with $T$ trials, random search explores $T$ distinct values *per dimension*, while grid search explores only $T^{1/d}$. For high-dimensional spaces with sparse importance, random search wins by orders of magnitude.

But random search uses **zero information from previous trials**. Trial 200 has no idea that trial 5 was disastrous and trial 47 was excellent.

### Bayesian optimization

Sequentially fit a **surrogate model** $p(y | \theta)$ to past observations, then choose the next $\theta$ to evaluate by maximizing an **acquisition function**. Each evaluation reduces uncertainty in unexplored regions and exploits known-good regions. The result is dramatic sample efficiency — typically 50–100 trials to reach the quality random search needs 1000+ for.


## 2. Sequential Model-Based Optimization (SMBO)

The generic SMBO loop:

```
history H ← ∅
while budget remains:
    fit surrogate S to H
    θ_next ← argmax_θ acquisition(θ; S)
    y_next ← f(θ_next)                  # the expensive evaluation
    H ← H ∪ {(θ_next, y_next)}
```

The two design choices are **the surrogate** (Gaussian Process, Random Forest, Parzen estimator, neural network) and **the acquisition function** (Expected Improvement, Upper Confidence Bound, Probability of Improvement, knowledge gradient).

### Expected Improvement (EI)

Given the best observed objective $y^\star$, the EI of $\theta$ is

$$
\mathrm{EI}(\theta) = \mathbb{E}\!\left[ \max\bigl(y^\star - f(\theta),\, 0\bigr) \right],
$$

where the expectation is taken under the surrogate's predictive distribution at $\theta$. Maximizing EI balances exploration (where $f$ is uncertain) and exploitation (where $f$ is predicted to be low). EI has a closed form under Gaussian surrogates and is the de-facto default for SMBO.


## 3. The Tree-structured Parzen Estimator (TPE)

TPE (Bergstra et al., 2011) inverts the modeling direction. Instead of fitting $p(y | \theta)$, it splits past observations at some quantile $y^\star$ and models

$$
p(\theta \mid y) = \begin{cases} \ell(\theta) & \text{if } y < y^\star, \\ g(\theta) & \text{if } y \ge y^\star, \end{cases}
$$

each density estimated by a Parzen-window (kernel density) estimator. Equivalently, $\ell$ is the density of "good" hyperparameters and $g$ is the density of "bad" ones.

### Deriving the acquisition

Expected Improvement under TPE's piecewise model has a closed form (Bergstra et al., 2011, §3):

$$
\mathrm{EI}_{y^\star}(\theta) = \int_{-\infty}^{y^\star} (y^\star - y) \, p(y \mid \theta) \, dy.
$$

Apply Bayes' rule $p(y \mid \theta) = p(\theta \mid y)\, p(y) / p(\theta)$ and use $p(\theta) = \gamma \ell(\theta) + (1-\gamma) g(\theta)$ where $\gamma = p(y < y^\star)$. After simplification:

$$
\boxed{\; \mathrm{EI}_{y^\star}(\theta) \propto \left( \gamma + \frac{g(\theta)}{\ell(\theta)} (1 - \gamma) \right)^{-1} \;}
$$

Maximizing EI is therefore equivalent to **maximizing the ratio**

$$
\frac{\ell(\theta)}{g(\theta)}.
$$

The next trial should be the $\theta$ that is most likely under the *good* distribution **relative to** the *bad* distribution.

### Why TPE excels for GBDT tuning

GBDT search spaces are:

- **High-dimensional** (often 8-12 parameters)
- **Heterogeneous** (mix of integer, float, log-scale, categorical)
- **Hierarchical** (some parameters only matter if another is set a certain way — e.g., `bagging_freq` only matters when `bagging_fraction < 1`)

TPE handles all three naturally because it models each parameter's density independently within $\ell$ and $g$, with explicit support for hierarchical search spaces.


In [ ]:
import numpy as np
import optuna
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'src'))

from gradient_forge.data.loaders import load_synthetic_classification
from gradient_forge.optimization import OptunaStudy, build_pruner
from gradient_forge.utils import seed_everything, Stopwatch
seed_everything(42)
optuna.logging.set_verbosity(optuna.logging.WARNING)


### 3.1 TPE on a 2D benchmark

The Branin function is a classic non-convex benchmark for black-box optimization. Run TPE for 80 trials and visualize where it places its evaluations.


In [ ]:
def branin(x, y):
    return (y - 5.1*x**2/(4*np.pi**2) + 5*x/np.pi - 6)**2            + 10*(1 - 1/(8*np.pi))*np.cos(x) + 10

def branin_objective(trial):
    x = trial.suggest_float("x", -5, 10)
    y = trial.suggest_float("y", 0, 15)
    return branin(x, y)

# TPE vs. random search.
study_tpe = optuna.create_study(direction="minimize",
                                sampler=optuna.samplers.TPESampler(seed=0))
study_rnd = optuna.create_study(direction="minimize",
                                sampler=optuna.samplers.RandomSampler(seed=0))
study_tpe.optimize(branin_objective, n_trials=80)
study_rnd.optimize(branin_objective, n_trials=80)

print(f"Best Branin value (lower is better):")
print(f"  random search : {study_rnd.best_value:.4f}")
print(f"  TPE           : {study_tpe.best_value:.4f}")


In [ ]:
# Plot the trial trajectories side by side.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Background: Branin landscape.
xs = np.linspace(-5, 10, 100); ys = np.linspace(0, 15, 100)
XX, YY = np.meshgrid(xs, ys)
ZZ = branin(XX, YY)

for ax, study, title in [(axes[0], study_rnd, "Random search"),
                          (axes[1], study_tpe, "TPE")]:
    ax.contourf(XX, YY, np.log(ZZ + 1), levels=20, alpha=0.4, cmap="viridis")
    xs_t = [t.params["x"] for t in study.trials]
    ys_t = [t.params["y"] for t in study.trials]
    ax.scatter(xs_t, ys_t, c=np.arange(len(xs_t)), cmap="Reds", s=30, edgecolors="black")
    ax.scatter(study.best_params["x"], study.best_params["y"],
               color="yellow", marker="*", s=400, edgecolors="black", label="best")
    ax.set_xlabel("x"); ax.set_ylabel("y")
    ax.set_title(f"{title} — best = {study.best_value:.3f}")
    ax.legend()
plt.tight_layout(); plt.show()


**Interpretation.** Random search scatters trials uniformly. TPE *clusters* its trials around the three global minima of Branin once it has identified them — and continues to occasionally explore the broader space (the few outliers).


## 4. Pruners — median and Hyperband

For iterative learners (boosting, neural nets), each trial reports intermediate scores after every cross-validation fold (or every epoch). **Pruners** abandon trials that are already trailing.

### MedianPruner

Maintain a running median of all completed trials' intermediate scores at each step. If a new trial's score at step $k$ is worse than the median, kill it. Simple, effective, default for most problems.

### HyperbandPruner

Bracket-based resource allocation (Li et al., 2017). Trials start with a small budget; the top $1/\eta$ survive to the next bracket with $\eta\times$ more budget. Continues geometrically until one trial uses the full budget. This is provably optimal up to log factors — kills under-performers aggressively while letting promising trials use full resources.

### Practical guidance

- Use **median pruning** for shallow search spaces and short trials.
- Use **Hyperband** when trials have a wide range of "right amounts of budget" — e.g. when `n_estimators` is in the search space.
- Set `n_warmup_steps` ≥ 5 so the pruner has data before killing.

### Quantifying the speed-up

Pruning typically discards 40–60% of trials with minimal loss in best-AUC discovery. The wall-clock saving is roughly proportional.


## 5. GPU acceleration

Tree-based models map well to GPU because histogram construction is a parallel reduction. Each library exposes a different flag:

| Library | CPU flag | GPU flag |
|---------|----------|----------|
| XGBoost | `device="cpu"` (default) | `device="cuda"` (requires CUDA build) |
| LightGBM | `device="cpu"` (default) | `device="gpu"` (requires CL/CUDA build) |
| CatBoost | `task_type="CPU"` (default) | `task_type="GPU"` (ships GPU binaries) |

### How GPU helps

1. **Histogram construction** is the dominant per-iteration cost. On CPU it is $O(n \cdot d)$ memory operations; on GPU it parallelizes across the $d$ axis.
2. **Split finding** for many features is independent → embarrassingly parallel.
3. **GPU memory** is a binding constraint: a 10M-row × 200-feature `float32` dataset is 8 GB, which fits comfortably on modern GPUs but is the upper end for laptop-class hardware.

### CUDA pseudo-flow

```
for each iteration:
    for each feature j  (parallel across GPU blocks):
        build histogram of (g, h) into K bins  (parallel reduction within block)
    find best (feature, bin) across all features  (parallel argmax across blocks)
    update leaf weights using w* = -G/(H+λ)
```

The single biggest win on GPU is that **modern GBDT libraries treat features as the parallel axis** — this is the reason GPU shines on wide tabular data.

### Our unified abstraction

The `OptunaStudy` driver in this repository accepts `gpu=True` and dispatches to the right device flag per library. The `XGBoostTrainer`, `LightGBMTrainer`, and `CatBoostTrainer` classes also accept the same boolean. Below we run on CPU because most environments lack CUDA — flip the flag locally to compare.


## 6. End-to-end Optuna study

Tune all three libraries on the same dataset. For each library:

1. Define a TPE search space (see `src/gradient_forge/optimization/search_spaces.py`).
2. Run 20 trials (bump to 200+ in production).
3. Use 3-fold CV inside each trial.
4. Median pruner kills under-performing trials early.
5. Best parameters and the full study object are returned.


In [ ]:
X, y = load_synthetic_classification(n_samples=8_000, n_features=25, random_state=42)

studies = {}
for algo in ["xgboost", "lightgbm", "catboost"]:
    print(f"\n=== Tuning {algo} ===")
    with Stopwatch() as sw:
        runner = OptunaStudy(
            algorithm=algo,
            task="binary",
            n_trials=20,                          # bump to 200+ in production
            cv_splits=3,
            gpu=False,                            # set True if CUDA is available
            random_state=42,
            pruner=build_pruner("median", n_warmup_steps=2),
        )
        studies[algo] = runner.optimize(X, y)
    n_pruned = sum(t.state == optuna.trial.TrialState.PRUNED for t in studies[algo].trials)
    print(f"  best CV AUC : {studies[algo].best_value:.4f}")
    print(f"  wall time   : {sw.seconds:.1f}s   ({len(studies[algo].trials)} trials, {n_pruned} pruned)")


In [ ]:
# Optimization-history plot for each study.
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (algo, study) in zip(axes, studies.items()):
    values = [t.value for t in study.trials if t.value is not None]
    if values:
        best_so_far = np.maximum.accumulate(values)
        ax.plot(values, "o-", alpha=0.4, label="per-trial AUC")
        ax.plot(best_so_far, "C3-", linewidth=2, label="best so far")
    ax.set_title(algo); ax.set_xlabel("trial"); ax.legend(); ax.grid(alpha=0.3)
axes[0].set_ylabel("CV AUC")
plt.tight_layout(); plt.show()


In [ ]:
# Inspect the best hyperparameters found.
rows = []
for algo, study in studies.items():
    rows.append({"algorithm": algo, "best_AUC": study.best_value, **study.best_params})
df_best = pd.DataFrame(rows).set_index("algorithm")
df_best


### 6.1 Parameter-importance analysis

Optuna can rank hyperparameters by their marginal effect on the objective (via fANOVA). This is invaluable for understanding *which* parameters matter most.


In [ ]:
# Compute parameter importance for the XGBoost study.
try:
    importance = optuna.importance.get_param_importances(studies["xgboost"])
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(list(importance.keys()), list(importance.values()), color="C0")
    ax.set_xlabel("relative importance")
    ax.set_title("XGBoost — parameter importance (fANOVA)")
    plt.tight_layout(); plt.show()
except (ValueError, RuntimeError) as e:
    print(f"Importance calculation needs more completed trials than we ran: {e}")


## 7. Persisting and resuming studies

Long-running tuning campaigns (200+ trials, multi-hour) should persist to disk so failures don't lose progress and so multiple workers can collaborate.

### SQLite-backed studies

```python
study = optuna.create_study(
    study_name="gradient_forge_v1",
    storage="sqlite:///optuna.db",
    load_if_exists=True,         # picks up where we left off
    sampler=optuna.samplers.TPESampler(),
    pruner=optuna.pruners.MedianPruner(),
)
```

### Distributed tuning

Multiple Python processes (or machines) can connect to the same SQLite (or PostgreSQL / MySQL) storage and run trials concurrently. Optuna handles the locking — each worker requests the next $\theta$ from the central database and reports the result back.

This is how production tuning systems run: a fleet of GPU workers all pulling from one study, each evaluating a different hyperparameter combination, with TPE coordinating the search globally.


In [ ]:
# Brief demonstration of persistent storage.
import os, tempfile
db_path = os.path.join(tempfile.gettempdir(), "demo_persist.db")
if os.path.exists(db_path):
    os.remove(db_path)
storage = f"sqlite:///{db_path}"

# First "session": 5 trials.
s1 = optuna.create_study(study_name="demo", storage=storage,
                         direction="minimize",
                         sampler=optuna.samplers.TPESampler(seed=0))
s1.optimize(branin_objective, n_trials=5)
print(f"Session 1 → completed {len(s1.trials)} trials, best = {s1.best_value:.4f}")

# Second "session": resume from disk, add 5 more.
s2 = optuna.create_study(study_name="demo", storage=storage,
                         load_if_exists=True, direction="minimize",
                         sampler=optuna.samplers.TPESampler(seed=0))
s2.optimize(branin_objective, n_trials=5)
print(f"Session 2 → resumed, now {len(s2.trials)} trials total, best = {s2.best_value:.4f}")

os.remove(db_path)


## 8. Exercises

1. **TPE vs. random — sample-efficiency curve.** Run both on the Branin function for trial counts $T \in \{20, 50, 100, 200, 500\}$ and plot the best-value-found vs. $T$. At what $T$ does TPE's advantage saturate?
2. **Multi-objective study.** Use `optuna.create_study(directions=["maximize", "minimize"])` to tune for *both* AUC (higher) and inference latency (lower). Plot the Pareto front.
3. **Hyperband vs. median.** Re-run the GBDT study with `build_pruner("hyperband")` and compare wall-clock time to median pruning.
4. **GPU benchmark.** If you have CUDA available, re-run Section 6 with `gpu=True` and report the wall-clock speed-up.

## Takeaways

- **TPE** is the default for GBDT tuning because it handles high-dimensional, heterogeneous, hierarchical search spaces natively.
- **Median pruning** discards 40-60% of trials with negligible loss in best-AUC discovery.
- The three libraries each expose a one-flag GPU switch — the `OptunaStudy` driver hides this behind a single boolean.
- **Persistent storage** turns tuning from a fragile foreground process into a durable, resumable, distributable campaign.

> **Next week:** assemble everything into a single production pipeline.
